In [7]:
import pandas as pd 
import numpy as np

In [ ]:
# read processed file
crsp = pd.read_csv('../data/processed/crsp_monthly_processed.csv')

In [11]:
# build reversal variable
crsp['rev'] = crsp.groupby('permno')['ret'].shift(1) 

In [12]:
crsp['rev'].head(10)

0         NaN
1   -0.047619
2    0.000000
3   -0.004286
4   -0.144928
5    0.067797
6    0.107302
7    0.072464
8   -0.027027
9    0.038333
Name: rev, dtype: float64

In [13]:
crsp.head(10)

,date,permno,ret,prc,shrout,cfacpr,cfacshr,vol,comnam,shrcd,exchcd,mkt_cap,rev
0,1994-01-31,10001,-0.047619,17.500,1091.0,3.0,3.0,150.0,ENERGY WEST INC,11,3,19092.500,NaN
1,1994-02-28,10001,0.000000,17.500,1091.0,3.0,3.0,58.0,ENERGY WEST INC,11,3,19092.500,-0.047619
2,1994-03-31,10001,-0.004286,17.250,1091.0,3.0,3.0,133.0,ENERGY WEST INC,11,3,18819.750,0.000000
3,1994-04-29,10001,-0.144928,14.750,1091.0,3.0,3.0,102.0,ENERGY WEST INC,11,3,16092.250,-0.004286
4,1994-05-31,10001,0.067797,15.750,1091.0,3.0,3.0,146.0,ENERGY WEST INC,11,3,17183.250,-0.144928
5,1994-06-30,10001,0.107302,8.625,2191.0,1.5,1.5,234.0,ENERGY WEST INC,11,3,18897.375,0.067797
6,1994-07-29,10001,0.072464,9.250,2191.0,1.5,1.5,259.0,ENERGY WEST INC,11,3,20266.750,0.107302
7,1994-08-31,10001,-0.027027,9.000,2191.0,1.5,1.5,159.0,ENERGY WEST INC,11,3,19719.000,0.072464
8,1994-09-30,10001,0.038333,9.250,2211.0,1.5,1.5,121.0,ENERGY WEST INC,11,3,20451.750,-0.027027
9,1994-10-31,10001,-0.054054,8.750,2211.0,1.5,1.5,117.0,ENERGY WEST INC,11,3,19346.250,0.038333


In [ ]:
import pandas as pd
import numpy as np

print("开始构建动量(MOM)与反转(REV)因子...\n")

# ==========================================
# 0. 极其重要：必须先按股票代码和时间排序
# ==========================================
# 任何时间序列的 shift 或 rolling 操作，前提都是数据绝对有序！
crsp = crsp.sort_values(by=['PERMNO', 'DATE']).reset_index(drop=True)

# ==========================================
# 1. 构建短期反转因子 (REV)
# ==========================================
# 逻辑：获取该股票上一行（上一个月）的总收益率
crsp['REV'] = crsp.groupby('PERMNO')['RET_TOTAL'].shift(1)

# ==========================================
# 2. 构建 12-1 动量因子 (MOM)
# ==========================================
# 技巧：连续相乘 (1+R) 在 Pandas 中容易遇到 NaN 对齐问题且速度慢。
# 数学转换：累积收益 = exp(sum(ln(1+R))) - 1。对数收益率相加等于简单收益率相乘。

# a. 将简单收益率转换为对数收益率
crsp['log_ret'] = np.log(1 + crsp['RET_TOTAL'])

# b. 计算过去 11 个月（包含当月）的累积对数收益
# min_periods=8 表示过去 11 个月中只要有 8 个月的有效数据，就计算该值，否则留空(NaN)
crsp['roll_11m_log'] = crsp.groupby('PERMNO')['log_ret'].transform(
    lambda x: x.rolling(window=11, min_periods=8).sum()
)

# c. 转换回简单累积收益率
crsp['cum_11m_ret'] = np.exp(crsp['roll_11m_log']) - 1

# d. 关键步：时间错位 (Shift)
# 我们在 t 月，需要的是 t-12 到 t-2 的累积收益。
# 上一步算出的 'cum_11m_ret' 在 t-2 那一行的值，正好代表了 t-12 到 t-2 的累积收益。
# 因此，我们将 'cum_11m_ret' 向下平移 2 行，赋给 t 月的 MOM 因子。
crsp['MOM'] = crsp.groupby('PERMNO')['cum_11m_ret'].shift(2)

# ==========================================
# 3. 清理中间变量
# ==========================================
crsp = crsp.drop(columns=['log_ret', 'roll_11m_log', 'cum_11m_ret'])

print("因子构建完成！\n")
print(crsp[['PERMNO', 'DATE', 'RET_TOTAL', 'REV', 'MOM']].head(15))

In [8]:
features = pd.read_csv('../data/processed/crsp_monthly_features.csv')

In [11]:
features.columns

Index(['date', 'permno', 'comnam', 'ret', 'ret_fwd_1m', 'prc', 'adj_prc',
       'shrout', 'mkt_cap', 'log_mkt_cap', 'ret_lag_1', 'ret_lag_2',
       'ret_lag_3', 'mom_2_6', 'mom_2_12', 'vol_3m', 'vol_6m', 'vol_12m',
       'price_ma_6_ratio', 'price_ma_12_ratio', 'turnover', 'turnover_3m',
       'turnover_12m', 'cfacpr', 'cfacshr', 'vol', 'shrcd', 'exchcd',
       'ret_lag_1_cs', 'ret_lag_2_cs', 'ret_lag_3_cs', 'mom_2_6_cs',
       'mom_2_12_cs', 'vol_3m_cs', 'vol_6m_cs', 'vol_12m_cs',
       'price_ma_6_ratio_cs', 'price_ma_12_ratio_cs', 'log_mkt_cap_cs',
       'turnover_cs', 'turnover_3m_cs', 'turnover_12m_cs'],
      dtype='object')

In [12]:
features.info

<bound method DataFrame.info of                date  permno           comnam       ret  ret_fwd_1m        prc  \
0        1994-01-31   10001  ENERGY WEST INC -0.047619    0.000000   17.50000   
1        1994-02-28   10001  ENERGY WEST INC  0.000000   -0.004286   17.50000   
2        1994-03-31   10001  ENERGY WEST INC -0.004286   -0.144928   17.25000   
3        1994-04-29   10001  ENERGY WEST INC -0.144928    0.067797   14.75000   
4        1994-05-31   10001  ENERGY WEST INC  0.067797    0.107302   15.75000   
...             ...     ...              ...       ...         ...        ...   
1354365  2024-08-30   93436        TESLA INC -0.077391    0.221942  214.11000   
1354366  2024-09-30   93436        TESLA INC  0.221942   -0.045025  261.63000   
1354367  2024-10-31   93436        TESLA INC -0.045025    0.381469  249.85001   
1354368  2024-11-29   93436        TESLA INC  0.381469    0.170008  345.16000   
1354369  2024-12-31   93436        TESLA INC  0.170008         NaN  403.84000

In [6]:
features.groupby('permno').count()

,date,comnam,ret,ret_fwd_1m,prc,adj_prc,shrout,mkt_cap,log_mkt_cap,ret_lag_1,...,price_ma_6_ratio,price_ma_12_ratio,turnover,turnover_3m,turnover_12m,cfacpr,cfacshr,vol,shrcd,exchcd
permno,,,,,,,,,,,,,,,,,,,,,
10001,283,283,283,282,283,283,283,283,283,282,...,277,271,283,280,271,283,283,283,283,283
10002,185,185,185,184,185,185,185,185,185,184,...,179,173,185,182,173,185,185,185,185,185
10003,3,3,3,2,3,3,3,3,3,2,...,0,0,3,0,0,3,3,3,3,3
10009,82,82,82,81,82,82,82,82,82,81,...,76,70,82,79,70,82,82,82,82,82
10010,16,16,16,15,16,16,16,16,16,15,...,10,4,16,13,4,16,16,16,16,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93432,8,8,8,7,8,8,8,8,8,7,...,2,0,8,5,0,8,8,8,8,8
93433,24,24,24,23,24,24,24,24,24,23,...,18,12,24,21,12,24,24,24,24,24
93434,36,36,36,35,36,36,36,36,36,35,...,30,24,36,33,24,36,36,36,36,36


In [14]:
features.ret_fwd_1m[0:10]

0    0.000000
1   -0.004286
2   -0.144928
3    0.067797
4    0.107302
5    0.072464
6   -0.027027
7    0.038333
8   -0.054054
9   -0.042857
Name: ret_fwd_1m, dtype: float64

In [15]:
features.ret

0         -0.047619
1          0.000000
2         -0.004286
3         -0.144928
4          0.067797
             ...   
1354365   -0.077391
1354366    0.221942
1354367   -0.045025
1354368    0.381469
1354369    0.170008
Name: ret, Length: 1354370, dtype: float64